# Load data

In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_json(
    '../data/raw/shopee_reviews_dataset.jsonl',
    lines=True
)
df.head()

,id,review,rating,label
0,74263765409,Hương vị:thom Chắc do bên giao hàng bị vỡ mấy...,3,negative
1,11104151002,Hương thơm:nhẹ nhàng Lợi ích:phục hồi cấp ẩm M...,5,positive
2,15888299382,Chất lượng sản phẩm:ok Đúng với mô tả:đúng Lầ...,5,positive
3,81030214453,Độ tuổi sử dụng:em bes Chất lượng sản phẩm:tot...,5,positive
4,88484377297,"Hương vị:Mix vị, Tím Mình mua 64 gói (32 gói ...",1,negative


In [5]:
print(df.shape)
print(df.info())

(9599, 4)
<class 'pandas.DataFrame'>
RangeIndex: 9599 entries, 0 to 9598
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      9599 non-null   int64
 1   review  9599 non-null   str  
 2   rating  9599 non-null   int64
 3   label   9599 non-null   str  
dtypes: int64(2), str(2)
memory usage: 2.0 MB
None


In [7]:
df_unaccented = pd.read_json(
    '../data/raw/aug_unaccented_reviews.jsonl',
    lines=True
)
df_unaccented.head()

,id,review,rating,label
0,59241444271,"Mui huong:Ngot nhe, thomm Phu hop voi loai da:...",4,positive
1,75292434640,Toi rat hai long voi dich vu cua nguoi ban va ...,5,positive
2,13364343551,Review nhe mot so dong sua con uong tang can t...,5,positive
3,84816190204,"Hieu qua:Sach Thiet ke:Dep Thiet ke nho gon, ...",5,positive
4,30102564756,"Giao hang nhanh, dong goi dep an toan Mui Pink...",4,positive


In [8]:
print(df_unaccented.shape)
print(df_unaccented.info())

(1348, 4)
<class 'pandas.DataFrame'>
RangeIndex: 1348 entries, 0 to 1347
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      1348 non-null   int64
 1   review  1348 non-null   str  
 2   rating  1348 non-null   int64
 3   label   1348 non-null   str  
dtypes: int64(2), str(2)
memory usage: 465.0 KB
None


In [11]:
print("NA counts:\n", df.isna().sum())
print("NA counts:\n", df_unaccented.isna().sum())

NA counts:
 id        0
review    0
rating    0
label     0
dtype: int64
NA counts:
 id        0
review    0
rating    0
label     0
dtype: int64


# Clean text

In [25]:
import re
import string

# syntax: re.sub(pattern, replace, text)
# find pattern and replace with 'replace'
# r"..." is raw string
# \S means character that  is not whitespace
# .*? means all characters
def remove_url(text):
    return re.sub(r"http\S+|www\S+", "", text)

def remove_html(text):
    return re.sub(r"<.*?>", "", text)

def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))
    # delete all punctuations

def remove_special_characters(text):
    return re.sub(r"[^a-zA-ZÀ-ỹ0-9\s]", "", text)

def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()
    # turn consecutive whitespaces into one, remove begin/end whitespace

def clean_text(text):
    text = text.lower()
    text = remove_url(text)
    text = remove_html(text)
    text = remove_punctuation(text)
    text = remove_special_characters(text)
    text = normalize_whitespace(text)

    return text

In [28]:
df["clean_review"] = df["review"].copy().apply(clean_text)
display(df.loc[0,"clean_review"])
display(df.loc[0,"review"])

'hương vịthom chắc do bên giao hàng bị vỡ mấy gói'

'Hương vị:thom  Chắc do bên giao hàng bị vỡ mấy gói'

In [30]:
df_unaccented["clean_review"] = df_unaccented["review"].copy().apply(clean_text)
display(df_unaccented.loc[0,"clean_review"])
display(df_unaccented.loc[0,"review"])

'mui huongngot nhe thomm phu hop voi loai daminh nghi la moi loai da cong dungduong trang nhung ma ch thay trang chi thay thom nhe helu mn minh feeedback dayy minh nghi ai cung muon co loi khuyen that long vay minh se co gang de mieu ta trai nghiem nhe minh thu nghiem trong mot tuan co ket hop ca scrub body dove tu unilever loai sakura luon sau thi minh thay hien tai no chi thom la chinh da minh cung deu mau hon nhung ma mau ngam ngam um minh nghi ve phan duong trang thi kbt tnao nhung minh nghi kh qua hieu qua tu hinh anh ban thay da minh truoc va sau day minh co cai vong thi n cung che nang phan nao nhung ma cam giac da de bat nang lam ngu day phong he he anh sang ma cam giac da ngam hon roi hic khong biet co phai do minh khong minh van boi kcn moi ngay nhe minh boi kha day dung loai senka duong am cong them ca ao chong nang ma van ngam mn biet s cach nao huong dan minh trang hon voi hic thi day la trai nghiem cua minh them nua muon nhan qua moi nguoi phai an du qua trong gio nhe nho

'Mui huong:Ngot nhe, thomm Phu hop voi loai da:minh nghi la moi loai da Cong dung:duong trang nhung ma ch thay trang chi thay thom nhe  Helu mn, minh feeedback dayy. Minh nghi ai cung muon co loi khuyen that long, vay minh se co gang de mieu ta trai nghiem nhe. Minh thu nghiem trong mot tuan, co ket hop ca scrub body dove tu unilever loai sakura luon. Sau thi minh thay hien tai no chi thom la chinh, da minh cung deu mau hon nhung ma mau ngam ngam . Um minh nghi ve phan duong trang thi kbt tnao nhung minh nghi kh qua hieu qua. Tu hinh anh ban thay da minh truoc va sau day. Minh co cai vong thi n cung che nang phan nao nhung ma cam giac da de bat nang lam. Ngu day phong he he anh sang ma cam giac da ngam hon roi hic khong biet co phai do minh khong. Minh van boi KCN MOI NGAY NHE. Minh boi kha day, dung loai senka duong am cong them ca ao chong nang ma van ngam. Mn biet s cach nao huong dan minh trang hon voi hic. Thi day la trai nghiem cua minh. Them nua muon nhan qua moi nguoi phai an d

# Normalize_text